# INF201 - Deliverable 2




In [1]:
"""User-supplied generator, adapted to a relative path and reproducible seed."""
import json
import random
from datetime import datetime, timedelta
from pathlib import Path

BASE_DIR = Path("data")
STORES = ["library_cafe", "science_building", "student_union"]
REGISTERS = ["POS_01", "POS_02", "POS_03"]
CURRENCIES = ["EUR", "NOK", "USD"]
ITEMS = [("Espresso", 3.00), ("Latte", 4.50), ("Cappuccino", 4.20),
         ("Muffin", 3.50), ("Croissant", 3.80), ("Sandwich", 7.50)]
STATUSES = ["SUCCESS"] * 4 + ["REFUND", "CANCELLED"]
NOISE_LINES = ["--- PRINTER PAPER LOW ---", "[DEBUG] BARCODE SCANNER REBOOTED",
               "=== NETWORK SYNC OK ==="]

def generate_log_line(ts: datetime, reg_id: str) -> str:
    """Generate one synthetic transaction line.
    
    Args:
        ts: Timestamp to include in the transaction.
        reg_id: Register identifier to include in the transaction.
    Returns:
        A newline-terminated log entry with randomized item, currency, price
        and status. The price has a 3 percent chance of being a 999.00 glitch."""
    item, base = random.choice(ITEMS)
    currency = random.choice(CURRENCIES)
    price = 999.0 if random.random() < 0.03 else round(base + random.uniform(-0.5, 0.5), 2)
    status = random.choice(STATUSES)
    return f"[{ts:%Y-%m-%d %H:%M:%S}] [{reg_id}] item={item} price={price:.2f} {currency} ({status})\n"

def create_dataset() -> None:
    """Write the reproducible synthetic dataset to BASE_DIR.
    
    Creates nine logs with 80 transactions each, alternating UTF-8 and Latin-1,
    plus nested store metadata. Seed 201 fixes the random sequence.
    Existing generated files are overwritten. Takes no arguments and returns
    None; filesystem errors propagate to the caller."""
    # Keep generated results reproducible across notebook runs.
    random.seed(201)
    for idx, (store, reg) in enumerate((s, r) for s in STORES for r in REGISTERS):
        folder = BASE_DIR / "2026" / store / reg
        folder.mkdir(parents=True, exist_ok=True)
        ts = datetime(2026, 10, 15, 7)
        lines = []
        for _ in range(80):
            ts += timedelta(seconds=random.randint(10, 180))
            if random.random() < 0.15:
                lines.append(random.choice(NOISE_LINES) + "\n")
            lines.append(generate_log_line(ts, reg))
        (folder / "register_sales.log").write_text(
            "".join(lines), encoding="utf-8" if idx % 2 == 0 else "iso-8859-1")
    config = {"company": "Campus Coffee", "regions": [{"region_name": "Main Campus",
        "stores": [{"name": "Library Cafe", "registers": [
            {"id": "POS_01", "active": True}, {"id": "POS_02", "active": True}]},
            {"name": "Science Building", "registers": [
            {"id": "POS_03", "active": True}, {"id": "POS_04", "active": False}]}]}]}
    (BASE_DIR / "store_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    print("Generated 9 logs with 720 transactions; random seed = 201.")


create_dataset()


Generated 9 logs with 720 transactions; random seed = 201.


In [2]:
"""INF201 Deliverable 2: ingestion, recursion, NumPy and Pandas."""
from pathlib import Path
import json
import re
import numpy as np
from numpy.typing import ArrayLike
import pandas as pd



## Task 1: encoding fallback and strict full-line parsing.

In [3]:
def read_log(path: str | Path) -> tuple[str, str]:
    """Read a log using UTF-8, falling back to Latin-1.
    
    Args:
        path: Path to the log file.
    Returns:
        A pair containing the decoded text and the encoding used.
    Raises:
        OSError: If the file cannot be opened or read."""
    path = Path(path)
    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"

LOG_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>POS_\d+)\] item=(?P<item>.+?) "
    r"price=(?P<price>\d+(?:\.\d+)?) (?P<currency>EUR|NOK|USD) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
)

def parse_log(text: str) -> list[dict[str, str | float]]:
    """Extract transactions from complete matching log lines.
    
    Args:
        text: Decoded log content, including possible system noise.
    Returns:
        Records with timestamp, register_id, item, price, currency and status.
        Prices are floats; the other fields are strings. Noise is ignored."""
    records = []
    for line in text.splitlines():
        match = LOG_PATTERN.fullmatch(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)
    return records



## Task 2: explicit recursion (no glob, rglob or os.walk).

In [4]:
def find_log_files(directory: str | Path) -> list[Path]:
    """Recursively collect log paths without glob or os.walk.
    
    Args:
        directory: Root folder to search.
    Returns:
        Deterministically ordered paths with a .log extension.
        Symbolic links are skipped to prevent recursive cycles.
    Raises:
        OSError: If a directory does not exist or cannot be read."""
    directory = Path(directory)
    found = []
    for entry in sorted(directory.iterdir()):
        if entry.is_symlink():
            continue  # Avoid recursive cycles.
        if entry.is_dir():
            found.extend(find_log_files(entry))
        elif entry.is_file() and entry.suffix.lower() == ".log":
            found.append(entry)
    return found

def active_register_ids(node: object) -> set[str]:
    """Collect active register IDs from nested JSON data.
    
    Args:
        node: A JSON value containing dictionaries, lists or scalar values.
    Returns:
        Unique string IDs whose associated active value is exactly True.
        Scalar values terminate recursion without adding any IDs."""
    found = set()
    if isinstance(node, dict):
        if node.get("active") is True and isinstance(node.get("id"), str):
            found.add(node["id"])
        for value in node.values():
            found.update(active_register_ids(value))
    elif isinstance(node, list):
        for value in node:
            found.update(active_register_ids(value))
    return found



## Task 3: population standard deviation and vectorized ratios.

In [5]:
def clean_price_outliers(prices: ArrayLike) -> np.ndarray:
    """Replace three-sigma outliers with the non-outlier median.
    
    Args:
        prices: Numeric array-like data, flattened into one dimension.
    Returns:
        A new one-dimensional float array; the input is not modified.
        Detection uses population standard deviation and one cleaning pass.
    Raises:
        ValueError: If values are non-finite or cannot be converted to floats."""
    values = np.asarray(prices, dtype=float).reshape(-1).copy()
    if values.size == 0:
        return values
    if not np.isfinite(values).all():
        raise ValueError("Prices must be finite.")
    # Build one mask before replacing values so detection uses original prices.
    outliers = np.abs(values - values.mean()) > 3 * values.std(ddof=0)
    if outliers.any():
        values[outliers] = np.median(values[~outliers])
    return values

def compute_refund_ratio(sales_count: np.ndarray,
                         refund_count: np.ndarray) -> np.ndarray:
    """Calculate element-wise refund-to-successful-sale ratios.
    
    Args:
        sales_count: Counts of successful sales, compatible with refund_count.
        refund_count: Counts of refunds, compatible with sales_count.
    Returns:
        Refund counts divided by sales counts plus 1e-6. Zero sales with
        positive refunds produce a large finite ratio, not a probability."""
    return np.asarray(refund_count, dtype=float) / (
        np.asarray(sales_count, dtype=float) + 1e-6)



## Task 4: timestamp index, hourly sums, status pivot and refund counts.

In [6]:
def build_dataframe(records: list[dict[str, str | float]]) -> pd.DataFrame:
    """Create a chronologically sorted transaction DataFrame.
    
    Args:
        records: Parsed records containing the six required transaction fields.
    Returns:
        A DataFrame indexed by timestamp, with floating-point prices.
    Raises:
        ValueError: If timestamps or prices cannot be converted."""
    columns = ["timestamp", "register_id", "item", "price", "currency", "status"]
    frame = pd.DataFrame(records, columns=columns)
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="raise")
    frame["price"] = frame["price"].astype(float)
    return frame.set_index("timestamp").sort_index(kind="stable")

def hourly_revenue(frame: pd.DataFrame) -> pd.DataFrame:
    """Sum recorded prices by item in one-hour windows.
    
    Args:
        frame: Transaction DataFrame with a DatetimeIndex and item/price columns.
    Returns:
        A table with item, timestamp and price columns. All statuses and
        currencies are included as specified; these are not net cash totals."""
    if frame.empty:
        return pd.DataFrame(columns=["item", "timestamp", "price"])
    return (frame.groupby("item")["price"].resample("1h").sum()
            .rename("price").reset_index())

def status_pivot(frame: pd.DataFrame) -> pd.DataFrame:
    """Summarize recorded transaction values by item and status.
    
    Args:
        frame: Transaction DataFrame containing item, status and price.
    Returns:
        An item-indexed table with SUCCESS, REFUND and CANCELLED columns.
        Missing categories contain zero; refund amounts remain positive."""
    return frame.pivot_table(index="item", columns="status", values="price",
                             aggfunc="sum", fill_value=0).reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"], fill_value=0)

def item_refund_ratios(frame: pd.DataFrame) -> pd.DataFrame:
    """Count successful sales and refunds for each item.
    
    Args:
        frame: Transaction DataFrame containing item and status columns.
    Returns:
        An item-indexed table with sales_count, refund_count and refund_ratio.
        CANCELLED transactions contribute to neither count."""
    counts = frame.groupby(["item", "status"]).size().unstack(fill_value=0)
    # Include absent statuses so items with no refunds still receive a ratio.
    counts = counts.reindex(columns=["SUCCESS", "REFUND"], fill_value=0)
    result = counts.rename(columns={"SUCCESS": "sales_count", "REFUND": "refund_count"})
    result["refund_ratio"] = compute_refund_ratio(
        result["sales_count"].to_numpy(), result["refund_count"].to_numpy())
    return result



## Task 5: assemble the workflow, filter inactive registers and write reports.

In [7]:
def run_pipeline(data_dir: str | Path, output_dir: str | Path) -> dict[str, pd.DataFrame | dict[str, object]]:
    """Read, clean and summarize the complete transaction dataset.
    
    Args:
        data_dir: Folder containing store_config.json and nested log files.
        output_dir: Destination for the four required reports and audit JSON.
    Returns:
        A dictionary containing transactions, hourly, pivot and ratios tables,
        plus an audit dictionary with record counts and encoding choices.
    Raises:
        OSError: If input files cannot be read or outputs cannot be written.
        ValueError: If configuration, timestamps or prices are invalid.
    
    Existing reports in output_dir are overwritten. Global outlier cleaning
    precedes NOK scale correction. Only active-register records are retained."""
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    # Create the report destination before writing output files.
    output_dir.mkdir(parents=True, exist_ok=True)
    config = json.loads((data_dir / "store_config.json").read_text(encoding="utf-8"))
    active = active_register_ids(config)
    paths = find_log_files(data_dir)
    records, encodings = [], {}
    for path in paths:
        content, encoding = read_log(path)
        encodings[str(path.relative_to(data_dir))] = encoding
        records.extend(parse_log(content))
    # Apply the global active-ID set before calculating any report.
    frame = build_dataframe([r for r in records if r["register_id"] in active])
    original = frame["price"].to_numpy(copy=True)
    frame["price"] = clean_price_outliers(original)
    replaced = int(np.count_nonzero(frame["price"].to_numpy() != original))
    # Repair NOK scale after outlier cleaning; this is not currency conversion.
    frame.loc[frame["currency"].eq("NOK"), "price"] *= 10.0
    hourly = hourly_revenue(frame)
    pivot = status_pivot(frame)
    ratios = item_refund_ratios(frame)
    frame.to_csv(output_dir / "cleaned_transactions.csv", float_format="%.6f")
    hourly.to_json(output_dir / "hourly_revenue.json", orient="records",
                   date_format="iso", indent=2)
    pivot.to_csv(output_dir / "status_pivot_table.csv", float_format="%.6f")
    ratios.to_csv(output_dir / "item_refund_ratios.csv", float_format="%.9f")
    audit = {"log_files": len(paths), "parsed_records": len(records),
             "retained_records": len(frame), "active_register_ids": sorted(active),
             "outliers_replaced": replaced, "encoding_used": encodings}
    (output_dir / "pipeline_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    return {"transactions": frame, "hourly": hourly, "pivot": pivot,
            "ratios": ratios, "audit": audit}



## Interpretation of the reports
Active register IDs are collected globally from the nested configuration and used to filter records. All three generated IDs are active, so all 720 transactions are retained. The configuration is not a store-to-register validation map; the requested set of IDs cannot express such a map.

The specified hourly sum and status pivot sum the recorded positive prices, including REFUND and CANCELLED. These are transaction-value summaries, not net cash revenue. Refund ratios use REFUND count divided by SUCCESS count plus 1e-6; cancelled transactions are excluded from both counts. With zero sales and positive refunds this formula yields a large finite number, not a probability.

NOK correction repairs a scale error; it does not convert currencies. The assignment requests item-only sums, which mix EUR, NOK and USD. They are reproduced as requested but cannot be interpreted as monetary totals in a single currency. Production financial reporting would group by currency or use documented exchange rates, and subtract refunds while excluding cancellations.

Outlier detection runs once over all uncorrected prices, before the NOK multiplication, using population standard deviation (ddof=0). Values more than three standard deviations from the mean are replaced with the median of non-outliers. This implements the specified global method; it may miss glitches in very small or heavily contaminated samples and does not preserve item-specific prices.


In [8]:
results = run_pipeline(Path("data"), Path("."))
print(json.dumps(results["audit"], indent=2))
print("\nFirst eight cleaned transactions:")
print(results["transactions"].head(8).to_string())

{
  "log_files": 9,
  "parsed_records": 720,
  "retained_records": 720,
  "active_register_ids": [
    "POS_01",
    "POS_02",
    "POS_03"
  ],
  "outliers_replaced": 15,
  "encoding_used": {
    "2026\\library_cafe\\POS_01\\register_sales.log": "utf-8",
    "2026\\library_cafe\\POS_02\\register_sales.log": "utf-8",
    "2026\\library_cafe\\POS_03\\register_sales.log": "utf-8",
    "2026\\science_building\\POS_01\\register_sales.log": "utf-8",
    "2026\\science_building\\POS_02\\register_sales.log": "utf-8",
    "2026\\science_building\\POS_03\\register_sales.log": "utf-8",
    "2026\\student_union\\POS_01\\register_sales.log": "utf-8",
    "2026\\student_union\\POS_02\\register_sales.log": "utf-8",
    "2026\\student_union\\POS_03\\register_sales.log": "utf-8"
  }
}

First eight cleaned transactions:
                    register_id        item  price currency     status
timestamp                                                             
2026-10-15 07:00:22      POS_01      Muffin

In [9]:
print("Hourly transaction values:")
print(results["hourly"].to_string(index=False))
print("\nStatus pivot:")
print(results["pivot"].to_string())
print("\nRefund ratios:")
print(results["ratios"].to_string())

Hourly transaction values:
      item           timestamp   price
Cappuccino 2026-10-15 07:00:00  968.85
Cappuccino 2026-10-15 08:00:00  628.29
Cappuccino 2026-10-15 09:00:00   96.93
 Croissant 2026-10-15 07:00:00  851.47
 Croissant 2026-10-15 08:00:00 1038.89
 Croissant 2026-10-15 09:00:00  137.27
  Espresso 2026-10-15 07:00:00  533.43
  Espresso 2026-10-15 08:00:00  692.58
  Espresso 2026-10-15 09:00:00   51.10
     Latte 2026-10-15 07:00:00  853.10
     Latte 2026-10-15 08:00:00 1101.84
     Latte 2026-10-15 09:00:00  136.98
    Muffin 2026-10-15 07:00:00  616.59
    Muffin 2026-10-15 08:00:00 1040.17
    Muffin 2026-10-15 09:00:00  115.73
  Sandwich 2026-10-15 07:00:00 1545.17
  Sandwich 2026-10-15 08:00:00 1697.26
  Sandwich 2026-10-15 09:00:00  108.11

Status pivot:
status      SUCCESS  REFUND  CANCELLED
item                                  
Cappuccino  1015.29  200.50     478.28
Croissant   1333.10  294.08     400.45
Espresso     936.00  136.39     204.72
Latte       1406.59  2

## Validation
Tests cover explicit recursion, UTF-8 and Latin-1 accented text, noise rejection, nested metadata, the strict boolean flag, outlier replacement, unchanged input arrays, empty and constant prices, zero-sales ratios, chronological sorting, hourly totals and missing status categories. The supplied generator emits only ASCII: both file encodings decode as UTF-8, so a successful UTF-8 decode does not identify the historical encoding. The separate accented-text fixture exercises the fallback.


In [10]:
"""Run with python -m unittest -v test_pipeline.py."""
import tempfile
import unittest
from pathlib import Path
import numpy as np
from numpy.typing import ArrayLike

class PipelineTests(unittest.TestCase):
    def test_encoding_parser_and_recursion(self) -> None:
        """Verify both encodings, noise rejection and nested log discovery."""
        with tempfile.TemporaryDirectory(dir=Path.cwd()) as folder:
            nested = Path(folder) / "a" / "b"
            nested.mkdir(parents=True)
            line = "[2026-10-15 08:30:00] [POS_01] item=Café Latte price=4.50 EUR (SUCCESS)"
            for name, encoding in [("one.log", "utf-8"), ("two.log", "iso-8859-1")]:
                path = nested / name
                path.write_text(line + "\n=== NETWORK SYNC OK ===", encoding=encoding)
                content, used = read_log(path)
                self.assertEqual(used, encoding)
                self.assertEqual(parse_log(content)[0]["item"], "Café Latte")
                self.assertEqual(len(parse_log(content)), 1)
            self.assertEqual(len(find_log_files(folder)), 2)
            self.assertEqual(parse_log(line + " unexpected"), [])

    def test_nested_metadata(self) -> None:
        """Verify nested traversal and require a literal True active flag."""
        node = [{"deep": [{"id": "POS_01", "active": True},
                          {"id": "POS_02", "active": False},
                          {"id": "POS_03", "active": 1}]}]
        self.assertEqual(active_register_ids(node), {"POS_01"})

    def test_numpy_edges(self) -> None:
        """Verify outlier replacement, input preservation and numerical edge cases."""
        prices = np.array([4.] * 50 + [999.])
        np.testing.assert_allclose(clean_price_outliers(prices), 4.)
        self.assertEqual(prices[-1], 999.)
        self.assertEqual(clean_price_outliers([]).shape, (0,))
        np.testing.assert_array_equal(clean_price_outliers([[3, 3]]), [3, 3])
        np.testing.assert_allclose(compute_refund_ratio(np.array([0, 2]),
                                                     np.array([0, 1])), [0, 1/2.000001])

    def test_reports(self) -> None:
        """Verify sorting, hourly sums, missing statuses and refund ratios."""
        rows = [{"timestamp": f"2026-10-15 {hour}:10:00", "register_id": "POS_01",
                 "item": "Latte", "price": price, "currency": "EUR", "status": status}
                for hour, price, status in [("09", 4., "REFUND"), ("08", 5., "SUCCESS")]]
        frame = build_dataframe(rows)
        self.assertTrue(frame.index.is_monotonic_increasing)
        self.assertEqual(hourly_revenue(frame)["price"].sum(), 9.)
        self.assertEqual(status_pivot(frame).loc["Latte", "CANCELLED"], 0)
        self.assertAlmostEqual(item_refund_ratios(frame).loc["Latte", "refund_ratio"], 1/1.000001)
        self.assertTrue(hourly_revenue(build_dataframe([])).empty)


import unittest
import io
stream = io.StringIO()
suite = unittest.defaultTestLoader.loadTestsFromTestCase(PipelineTests)
run = unittest.TextTestRunner(stream=stream, verbosity=2).run(suite)
print(stream.getvalue())
assert run.wasSuccessful()
assert results["audit"]["parsed_records"] == 720
assert results["audit"]["retained_records"] == 720
assert not (results["transactions"]["price"] == 999).any()
assert np.isclose(results["hourly"]["price"].sum(), results["transactions"]["price"].sum())
print("End-to-end row counts, glitch removal and aggregation conservation: PASS")

test_encoding_parser_and_recursion (__main__.PipelineTests.test_encoding_parser_and_recursion)
Verify both encodings, noise rejection and nested log discovery. ... ok
test_nested_metadata (__main__.PipelineTests.test_nested_metadata)
Verify nested traversal and require a literal True active flag. ... ok
test_numpy_edges (__main__.PipelineTests.test_numpy_edges)
Verify outlier replacement, input preservation and numerical edge cases. ... ok
test_reports (__main__.PipelineTests.test_reports)
Verify sorting, hourly sums, missing statuses and refund ratios. ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.196s

OK

End-to-end row counts, glitch removal and aggregation conservation: PASS


## Results and limitations
The workflow found 9 logs, parsed 720 transactions and replaced 15 outliers. All records belong to active registers. The output contains six item types. The largest observed refund-to-success ratio belongs to Sandwich, at 0.325. These are synthetic, seed-dependent findings, not actual campus trading results.

The reader propagates permission/missing-file errors rather than silently dropping data. Latin-1 can decode any byte sequence, so the fallback assumes only the two stated encodings. The regex enforces whole-line structure; calendar validity is subsequently checked by Pandas. No timestamp timezone is provided. Symlink directories are skipped to avoid recursion cycles.

## GenAI use disclosure
we used ai to help with the code and to understand the tasks.

## Submission items requiring the group
No GitHub repository URL was provided, and no repository or commit history was fabricated. Add the actual repository link and generate git_log.txt after the final commit using the command in README.md. Record the five-minute video with contributions from both group members using VIDEO_OUTLINE.md. Actual recording and submission remain for the group.
